# Predicting Gene Expression from Sequence Features — A Machine Learning Approach

**Author:** Samuel Ukwueze  
**Date:** February 2026  

---

One question I keep coming back to in computational biology is how much of gene expression we can predict just from sequence features — without any prior biological knowledge baked in. This notebook explores that using a supervised ML approach.

The setup: I'm simulating a dataset of gene promoter sequences with associated expression levels, then extracting sequence features (k-mer composition, GC content, motif presence) and training a few models to see what sticks. The point isn't to build something publication-ready — it's to build a clean, interpretable pipeline that could be adapted to real promoter data.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from itertools import product
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

np.random.seed(21)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

print('All imports loaded.')

## 2. Simulating Promoter Sequences and Expression Data

I'm generating 300 synthetic promoter sequences (200bp each) with expression levels that are influenced by a few sequence properties — GC content, presence of a TATA-box motif, and a couple of k-mer frequencies. This mimics the kind of signal you'd expect in real promoter biology, just in a controlled way.

In [ ]:
def generate_promoter(length=200, gc_bias=0.5, add_tata=False):
    at = (1 - gc_bias) / 2
    gc = gc_bias / 2
    seq = list(np.random.choice(['A', 'T', 'G', 'C'], size=length, p=[at, at, gc, gc]))
    if add_tata:
        # Insert TATAAA at position ~-30 (position 170 in 200bp sequence)
        tata_pos = np.random.randint(160, 175)
        for j, base in enumerate('TATAAA'):
            seq[tata_pos + j] = base
    return ''.join(seq)

n_genes = 300
gc_contents = np.random.uniform(0.30, 0.70, n_genes)
has_tata = np.random.binomial(1, 0.4, n_genes).astype(bool)

promoters = [generate_promoter(200, gc, tata) for gc, tata in zip(gc_contents, has_tata)]

# Expression as a function of sequence properties + noise
expression = (
    3.0 * gc_contents +
    1.5 * has_tata.astype(float) +
    np.random.normal(0, 0.4, n_genes)
)
expression = np.clip(expression, 0, None)  # no negative expression

print(f'Generated {n_genes} promoter sequences (200bp each)')
print(f'Expression range: {expression.min():.2f} — {expression.max():.2f}')
print(f'Sequences with TATA box: {has_tata.sum()} ({has_tata.mean()*100:.0f}%)')

## 3. Feature Extraction

This is where most of the biological thinking goes. I'm extracting three types of features from each promoter sequence: GC content, all 2-mer frequencies (16 features), and binary flags for known regulatory motifs. Together these give the model something meaningful to work with.

In [ ]:
# All possible 2-mers
all_dimers = [''.join(p) for p in product('ACGT', repeat=2)]

# Regulatory motifs to scan for
motifs = {
    'TATAAA': 'TATA_box',
    'CCAAT':  'CCAAT_box',
    'GGGCGG': 'GC_box',
    'CACGTG': 'E_box',
    'TGAGTC': 'AP1_site'
}

def extract_features(seq):
    seq = seq.upper()
    total = len(seq)
    features = {}

    # GC content
    features['gc_content'] = (seq.count('G') + seq.count('C')) / total

    # 2-mer frequencies
    dimer_counts = Counter([seq[i:i+2] for i in range(len(seq)-1)])
    dimer_total = sum(dimer_counts.values())
    for dimer in all_dimers:
        features[f'dimer_{dimer}'] = dimer_counts.get(dimer, 0) / dimer_total

    # Motif presence
    for motif, name in motifs.items():
        features[f'motif_{name}'] = int(motif in seq)

    return features

feature_list = [extract_features(seq) for seq in promoters]
X = pd.DataFrame(feature_list)
y = expression

print(f'Feature matrix shape: {X.shape}')
print(f'Features: {list(X.columns[:5])} ... and {X.shape[1]-5} more')
X.head(3)

## 4. Exploratory Look at Features vs Expression

Before running any models, I want to see how individual features correlate with expression. This gives a rough sense of which features actually carry signal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# GC vs expression
axes[0].scatter(X['gc_content'], y, alpha=0.5, color='#3498DB', s=20, edgecolors='none')
z = np.polyfit(X['gc_content'], y, 1)
p = np.poly1d(z)
xline = np.linspace(X['gc_content'].min(), X['gc_content'].max(), 100)
axes[0].plot(xline, p(xline), color='#E74C3C', linewidth=1.5)
axes[0].set_xlabel('GC Content')
axes[0].set_ylabel('Expression Level')
axes[0].set_title('GC Content vs Expression')

# TATA box vs expression
tata_flag = X['motif_TATA_box'].astype(bool)
axes[1].boxplot([y[~tata_flag], y[tata_flag]], labels=['No TATA', 'TATA Present'],
                patch_artist=True,
                boxprops=dict(facecolor='#2ECC71', alpha=0.7))
axes[1].set_ylabel('Expression Level')
axes[1].set_title('TATA Box vs Expression')

# Correlation of all features with expression
correlations = X.corrwith(pd.Series(y)).abs().sort_values(ascending=False).head(15)
axes[2].barh(correlations.index[::-1], correlations.values[::-1],
             color='#9B59B6', edgecolor='white', linewidth=0.5)
axes[2].set_xlabel('|Pearson Correlation| with Expression')
axes[2].set_title('Top 15 Features by Correlation')

plt.tight_layout()
plt.show()

## 5. Model Training and Comparison

I'm comparing four models — Ridge regression (linear baseline), ElasticNet (sparse linear), Random Forest, and Gradient Boosting. Cross-validation R² is the primary metric I care about here since it tells me how much of the expression variance the model actually explains.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

models = {
    'Ridge':            Ridge(alpha=1.0),
    'ElasticNet':       ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000),
    'Random Forest':    RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42),
    'Gradient Boost':   GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                                   learning_rate=0.05, random_state=42)
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    X_use = X_train_sc if name in ['Ridge', 'ElasticNet'] else X_train.values
    X_te  = X_test_sc  if name in ['Ridge', 'ElasticNet'] else X_test.values

    cv_scores = cross_val_score(model, X_use, y_train, cv=cv, scoring='r2')
    model.fit(X_use, y_train)
    y_pred = model.predict(X_te)

    results[name] = {
        'cv_r2_mean': cv_scores.mean(),
        'cv_r2_std':  cv_scores.std(),
        'test_r2':    r2_score(y_test, y_pred),
        'test_rmse':  np.sqrt(mean_squared_error(y_test, y_pred)),
        'model':      model,
        'y_pred':     y_pred,
        'X_te':       X_te
    }
    print(f"{name:<18} CV R²: {cv_scores.mean():.3f} ± {cv_scores.std():.3f} "
          f"| Test R²: {r2_score(y_test, y_pred):.3f} "
          f"| RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")

## 6. Predicted vs Actual — Visual Check

Numbers alone don't tell the whole story. I want to see how predictions track across the expression range — whether there's systematic bias at the high or low end, which is common with tree-based models.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes = axes.flatten()
colors = ['#3498DB', '#2ECC71', '#E74C3C', '#9B59B6']

for idx, (name, res) in enumerate(results.items()):
    ax = axes[idx]
    ax.scatter(y_test, res['y_pred'], alpha=0.6, color=colors[idx], s=25, edgecolors='none')

    min_val = min(y_test.min(), res['y_pred'].min())
    max_val = max(y_test.max(), res['y_pred'].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1, alpha=0.6)

    ax.set_xlabel('Actual Expression')
    ax.set_ylabel('Predicted Expression')
    ax.set_title(f'{name}\nR² = {res["test_r2"]:.3f} | RMSE = {res["test_rmse"]:.3f}')

plt.suptitle('Predicted vs Actual Expression — All Models', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 7. Feature Importance — Random Forest

Random Forest gives us built-in feature importance, but I prefer using permutation importance since it's less biased toward high-cardinality features. This tells us which sequence features the model actually relies on most.

In [ ]:
rf_model = results['Random Forest']['model']
X_te_rf  = results['Random Forest']['X_te']

perm_imp = permutation_importance(rf_model, X_te_rf, y_test,
                                   n_repeats=20, random_state=42)

imp_df = pd.DataFrame({
    'feature':    X.columns,
    'importance': perm_imp.importances_mean,
    'std':        perm_imp.importances_std
}).sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1],
        xerr=imp_df['std'][::-1], color='#E74C3C',
        edgecolor='white', linewidth=0.5, capsize=3)
ax.set_xlabel('Permutation Importance (Mean Decrease in R²)')
ax.set_title('Top 20 Features — Random Forest Permutation Importance')
plt.tight_layout()
plt.show()

print('Top 5 features:')
print(imp_df[['feature', 'importance']].head(5).to_string(index=False))

## 8. Model Comparison Summary

Pulling everything together into one clean comparison.

In [ ]:
summary_df = pd.DataFrame({
    name: {
        'CV R² (mean)': f"{res['cv_r2_mean']:.3f}",
        'CV R² (±std)': f"± {res['cv_r2_std']:.3f}",
        'Test R²':      f"{res['test_r2']:.3f}",
        'Test RMSE':    f"{res['test_rmse']:.3f}"
    }
    for name, res in results.items()
}).T

print('Model Performance Summary')
print('=' * 60)
print(summary_df.to_string())

# Bar chart of test R²
test_r2s = {name: res['test_r2'] for name, res in results.items()}
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(test_r2s.keys(), test_r2s.values(),
              color=colors, edgecolor='white', linewidth=0.8)
ax.set_ylabel('Test R²')
ax.set_title('Model Comparison — Test Set R²')
ax.set_ylim(0, 1)
for bar, val in zip(bars, test_r2s.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

## 9. Reflections

The tree-based models predictably outperformed the linear ones here, partly because the relationship between sequence features and expression isn't strictly linear. The permutation importance results also made biological sense — GC content and TATA box presence came up as the most informative features, which is exactly what we built into the simulation.

With real promoter data, I'd go a few steps further — include position-specific features (where in the promoter a motif sits matters a lot), use one-hot encoded sequences as input to a convolutional model, and do proper hyperparameter tuning via grid search or Optuna. But as a starting framework for thinking about sequence-to-expression prediction, this holds up pretty well.

---
*Analysis performed in Python 3.11 | Libraries: NumPy, Pandas, Scikit-learn, Matplotlib, Seaborn*